# IM Session 30 — LLM APIs, Integration & Ethics

**Batch:** IITP-AIMLTN-2605  |  **Curriculum session:** 12.2  |  **Module 3: GenAI & Agents**

## Learning Objectives

By the end of this notebook you will be able to:

- **LO 1** — Describe the anatomy of an LLM API call and handle API keys safely
- **LO 2** — Apply production usage patterns: timeouts, retries with backoff, batching and cost estimation
- **LO 3** — Build input and output guardrails, and state the Terms-of-Service and ethical limits that bind you

## The situation

Session 29 built prompts by hand and priced them. Today the same triage job has to run **programmatically**, over
the same 40 support tickets, without a human pasting anything.

That change introduces every problem this session is about: a secret that must not leak, a network that will
fail, a bill that grows per token, personal data that must not be sent carelessly, and a reply that arrives as
text when your code expects structure.

> **Runs completely offline — no API key, no network, no cost.**
> Calling a real provider in class would need a paid key, would fail on a weak connection, and would give
> different output on every run. So we use a **deterministic simulator** with the *same request and response
> shape* as the real thing. Every real SDK call is shown too, as a clearly-marked reference block you can copy
> once you have your own key.


## Setup

In [1]:
import json
import os
import re
import time
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 60)

tickets = pd.read_csv(Path("..") / "Datasets" / "support_tickets.csv")
CATEGORIES = sorted(tickets["category"].unique())

WORD_RE = re.compile(r"\w+|[^\w\s]")


def count_tokens(text, chunk=4):
    """Same approximate tokenizer as session 29, restated so this notebook stands alone."""
    n = 0
    for piece in WORD_RE.findall(str(text)):
        n += 1 if len(piece) <= chunk else -(-len(piece) // chunk)
    return n


print(f"tickets    : {len(tickets)}")
print(f"categories : {CATEGORIES}")

tickets    : 40
categories : ['account', 'billing', 'delivery', 'how_to', 'product_issue', 'refund']


---

## Part 1 — Anatomy of an API call

Every major provider exposes the same four ideas, whatever the SDK looks like:

| Piece | What it carries |
|---|---|
| **model** | Which model to run |
| **messages** | The conversation so far, as a list of `{role, content}` — roles are `system`, `user`, `assistant` |
| **parameters** | `temperature`, `max_tokens`, and similar controls |
| **response** | The generated text, plus a `usage` block telling you what it cost |

Here is the real call for two providers. **These cells are reference only — we do not run them**, because they
need a paid key and a network.

```python
# OpenAI - reference only, needs `pip install openai` and a real key
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Ticket: {text}\nCategory:"},
    ],
    temperature=0,
    max_tokens=10,
)
print(response.choices[0].message.content)
print(response.usage.prompt_tokens, response.usage.completion_tokens)
```

```python
# Google Gemini - reference only, needs `pip install google-generativeai` and a real key
import google.generativeai as genai

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
model = genai.GenerativeModel("gemini-1.5-flash", system_instruction=SYSTEM_PROMPT)
result = model.generate_content(
    f"Ticket: {text}\nCategory:",
    generation_config={"temperature": 0, "max_output_tokens": 10},
)
print(result.text)
```

Different names, identical shape: a system instruction, a user message, decoding parameters, and a response
carrying text plus usage. Learn the shape once and switching provider is a small change.

### The simulator

Below is our stand-in. It is a **rule-based keyword matcher**, not a language model — but it accepts and returns
exactly the structure above, so all the integration code we write around it is the real thing.

It is deliberately imperfect: on some tickets it returns a value that is *not* a valid category. That is not a
bug in the notebook, it is the situation your output guardrail in Part 4 has to survive.

In [2]:
KEYWORDS = {
    "refund":        ["refund", "money back", "reversal", "credited", "cancelled"],
    "delivery":      ["delivery", "deliver", "shipped", "courier", "parcel", "tracking", "dispatch", "pincode"],
    "billing":       ["charged", "invoice", "gst", "emi", "coupon", "debit", "payment", "billing"],
    "product_issue": ["defective", "cracked", "broken", "missing", "wrong", "hot", "sparked", "bubbles"],
    "how_to":        ["how do i", "how many", "can i", "is there", "where can"],
    "account":       ["log in", "login", "password", "account", "merge", "delete my account"],
}


class MockLLM:
    """Deterministic stand-in with the same request/response shape as a real chat API."""

    def __init__(self, fail_every=0):
        self.fail_every = fail_every      # simulate transient network errors
        self.calls = 0

    def chat(self, model, messages, temperature=0, max_tokens=10, timeout=10):
        self.calls += 1
        if self.fail_every and self.calls % self.fail_every == 0:
            raise TimeoutError("upstream timed out")

        user = messages[-1]["content"].lower()
        scores = {c: sum(k in user for k in kws) for c, kws in KEYWORDS.items()}
        best = max(scores, key=scores.get)
        # No keyword matched -> return something outside the allowed set, on purpose.
        content = best if scores[best] > 0 else "Sorry, I am not sure about this one."

        prompt_tokens = sum(count_tokens(m["content"]) for m in messages)
        return {
            "model": model,
            "choices": [{"message": {"role": "assistant", "content": content},
                         "finish_reason": "stop"}],
            "usage": {"prompt_tokens": prompt_tokens,
                      "completion_tokens": count_tokens(content),
                      "total_tokens": prompt_tokens + count_tokens(content)},
        }


SYSTEM_PROMPT = (
    "You are a support ticket triage assistant for an Indian e-commerce company.\n"
    f"Classify each ticket into exactly one category: {', '.join(CATEGORIES)}.\n"
    "Reply with the category name only, in lowercase. No explanation."
)

llm = MockLLM()
demo = tickets.iloc[1]["ticket_text"]
resp = llm.chat(
    model="mock-triage-1",
    messages=[{"role": "system", "content": SYSTEM_PROMPT},
              {"role": "user", "content": f"Ticket: {demo}\nCategory:"}],
)
print(json.dumps(resp, indent=2))

{
  "model": "mock-triage-1",
  "choices": [
    {
      "message": {
        "role": "assistant",
        "content": "delivery"
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 105,
    "completion_tokens": 2,
    "total_tokens": 107
  }
}


**Interpretation.** Note the `usage` block. Real providers bill on exactly these two numbers, and they are the
only reliable basis for a cost estimate — not your own guess about prompt length.

---

## Part 2 — Managing API keys

An API key is a **password that spends your money**. The rules are short and non-negotiable:

1. Never write a key in a notebook, a script, or anything git tracks.
2. Load it from an environment variable or a secrets manager.
3. Never print it, log it, or put it in an error message.
4. Rotate it immediately if it is ever exposed — assume any key that touched a repo is burned.

The single most common breach in student projects is a key committed to GitHub in a notebook cell. Public repos
are scanned by bots within minutes.

In [3]:
def load_api_key(var_name="OPENAI_API_KEY"):
    """Load a key from the environment, failing safely and without ever revealing it."""
    key = os.environ.get(var_name)
    if not key:
        raise RuntimeError(
            f"{var_name} is not set. Export it in your shell before running:\n"
            f"    export {var_name}='your-key-here'"
        )
    return key


def mask(key):
    """What is safe to print in a log."""
    return f"{key[:3]}...{key[-2:]}  (length {len(key)})" if len(key) > 6 else "***"


try:
    api_key = load_api_key()
    print("key loaded:", mask(api_key))
except RuntimeError as err:
    print("SAFE FAILURE - no key present, and nothing secret was printed:\n")
    print(err)

SAFE FAILURE - no key present, and nothing secret was printed:

OPENAI_API_KEY is not set. Export it in your shell before running:
    export OPENAI_API_KEY='your-key-here'


**Interpretation.** The failure path is the point. It says exactly what to do, and it does not print, guess or
partially reveal the secret. If you had a key set, `mask()` shows the most you should ever write to a log — just
enough to confirm *which* key is loaded, never enough to use it.

In [4]:
# What NOT to do - shown so it is recognisable in review, never so it is copied.
BAD_EXAMPLES = [
    'api_key = "sk-proj-abc123realkeyhere"        # hardcoded in the notebook',
    'print(f"calling with key {api_key}")          # leaked into stdout and logs',
    'requests.get(f"https://api.x.com?key={api_key}")   # secret in a URL, logged by every proxy',
    'except Exception as e: print(e, api_key)      # leaked in an error handler',
]
for line in BAD_EXAMPLES:
    print("  WRONG:", line)

  WRONG: api_key = "sk-proj-abc123realkeyhere"        # hardcoded in the notebook
  WRONG: print(f"calling with key {api_key}")          # leaked into stdout and logs
  WRONG: requests.get(f"https://api.x.com?key={api_key}")   # secret in a URL, logged by every proxy
  WRONG: except Exception as e: print(e, api_key)      # leaked in an error handler


The third one catches people out most often. Query strings are recorded by proxies, load balancers and browser
history, so a key in a URL is a key in somebody else's log file. Credentials belong in a header or an SDK
client, never in a URL.

---

## Part 3 — Usage patterns that survive production

Three things are true of every network API: it will be **slow** sometimes, it will **fail** sometimes, and it
**charges you** every time. Handle all three deliberately.

### Retries with exponential backoff

Retry only transient failures — timeouts, 429 rate limits, 5xx. Never retry a 400 (your request is malformed) or
a 401 (your key is wrong): those will fail identically forever. Back off exponentially so you do not add load to
a service that is already struggling, and always cap the attempts.

In [5]:
def call_with_retry(client, messages, max_attempts=4, base_delay=0.01, model="mock-triage-1"):
    """Retry transient failures with exponential backoff. Returns (response, attempts_used)."""
    for attempt in range(1, max_attempts + 1):
        try:
            return client.chat(model=model, messages=messages, temperature=0), attempt
        except (TimeoutError, ConnectionError):
            if attempt == max_attempts:
                raise                                  # give up honestly, do not return a fake result
            time.sleep(base_delay * (2 ** (attempt - 1)))   # 1x, 2x, 4x ...
    raise RuntimeError("unreachable")


flaky = MockLLM(fail_every=2)          # every 2nd call times out
msgs = [{"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Ticket: {demo}\nCategory:"}]

for i in range(3):
    resp, attempts = call_with_retry(flaky, msgs)
    print(f"request {i + 1}: succeeded on attempt {attempts} -> "
          f"{resp['choices'][0]['message']['content']}")

request 1: succeeded on attempt 1 -> delivery
request 2: succeeded on attempt 2 -> delivery
request 3: succeeded on attempt 2 -> delivery


**Interpretation.** Every request still succeeded despite half the underlying calls failing — but notice the
function **re-raises** once attempts are exhausted. A retry wrapper that silently returns a default on final
failure is worse than no wrapper, because the caller cannot tell a real answer from a fallback.

### Cost estimation before you run

Estimate the bill *before* launching a batch, not after. Rates below are **illustrative placeholders** — always
check the provider's current pricing page, since these change.

In [6]:
# Illustrative only - not live pricing. Replace with the provider's current published rates.
RATE_INPUT_PER_1K = 0.00015     # USD per 1K input tokens
RATE_OUTPUT_PER_1K = 0.00060    # USD per 1K output tokens
EXPECTED_OUTPUT_TOKENS = 5      # the reply is one short category word

system_cost = count_tokens(SYSTEM_PROMPT)
per_ticket = [system_cost + count_tokens(f"Ticket: {t}\nCategory:") for t in tickets["ticket_text"]]

total_input = sum(per_ticket)
total_output = EXPECTED_OUTPUT_TOKENS * len(tickets)
usd = total_input / 1000 * RATE_INPUT_PER_1K + total_output / 1000 * RATE_OUTPUT_PER_1K

print(f"input tokens for all {len(tickets)} tickets : {total_input:,}")
print(f"output tokens (estimated)          : {total_output:,}")
print(f"estimated cost for one full pass   : ${usd:.5f}")
print(f"cost per 100,000 tickets           : ${usd / len(tickets) * 100_000:.2f}")

input tokens for all 40 tickets : 4,307
output tokens (estimated)          : 200
estimated cost for one full pass   : $0.00077
cost per 100,000 tickets           : $1.92


**Interpretation.** Fractions of a cent for this batch — but the right-hand figure is the one that matters when
someone proposes running it over a year of tickets. Note also that the system prompt is re-sent on every single
request: it is **74 tokens** paid 40 times here — 68.7% of all input tokens — which is why trimming a system
prompt is usually the cheapest optimisation available.

---

## Part 4 — Guardrails

Guardrails come in two halves, and you need both.

- **Input guardrails** control what leaves your system. Chiefly: do not send personal data you do not need.
- **Output guardrails** control what you accept back. The model returns free text; your code expects structure.

### Input side: redact PII before sending

Our tickets contain real-shaped personal data — email addresses, phone numbers, order IDs. The triage model
needs none of it to pick a category, so none of it should be sent. This is **data minimisation**, and under
India's DPDP Act it is a legal expectation, not merely good manners.

In [7]:
PII_PATTERNS = {
    "EMAIL": re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),
    "PHONE": re.compile(r"\b[6-9]\d{9}\b"),
    "ORDER_ID": re.compile(r"\bMRD-\d{5}\b"),
}


def redact(text):
    """Replace personal identifiers with type tags. Returns (clean_text, what_was_found)."""
    found = {}
    clean = str(text)
    for tag, pattern in PII_PATTERNS.items():
        hits = pattern.findall(clean)
        if hits:
            found[tag] = len(hits)
            clean = pattern.sub(f"[{tag}]", clean)
    return clean, found


hits_total = {"EMAIL": 0, "PHONE": 0, "ORDER_ID": 0}
tickets_with_pii = 0
for text in tickets["ticket_text"]:
    _, found = redact(text)
    if found:
        tickets_with_pii += 1
        for tag, n in found.items():
            hits_total[tag] += n

print(f"tickets containing PII : {tickets_with_pii} of {len(tickets)}")
print(f"identifiers found      : {hits_total}\n")

example = tickets.loc[tickets["ticket_id"] == "TKT-1001", "ticket_text"].iloc[0]
clean, found = redact(example)
print("BEFORE:", example)
print("\nAFTER :", clean)
print("\nremoved:", found)

tickets containing PII : 8 of 40
identifiers found      : {'EMAIL': 3, 'PHONE': 2, 'ORDER_ID': 5}

BEFORE: Ordered a wireless keyboard on 28 July, delivered cracked. I want a refund not a replacement. Order MRD-88213. Reach me at arun.k@example.com

AFTER : Ordered a wireless keyboard on 28 July, delivered cracked. I want a refund not a replacement. Order [ORDER_ID]. Reach me at [EMAIL]

removed: {'EMAIL': 1, 'ORDER_ID': 1}


**Interpretation.** Ten identifiers across eight tickets never leave the building, and the category is still
perfectly inferable from what remains. That is the test for any redaction rule: **does removing it cost you any
accuracy on the actual task?** If not, there was never a reason to send it.

### Output side: validate before you trust

The model returns a string. Your pipeline needs one of six known categories. Those are not the same thing, and
the gap is where production breaks.

In [8]:
def validate_category(raw, allowed=CATEGORIES):
    """Normalise and check the model's reply. Never let an unvalidated value into the pipeline."""
    value = str(raw).strip().lower().strip(".")
    if value in allowed:
        return {"ok": True, "category": value, "reason": None}
    return {"ok": False, "category": None, "reason": f"not an allowed category: {raw!r}"}


records = []
client = MockLLM()
for row in tickets.itertuples():
    clean, _ = redact(row.ticket_text)                       # input guardrail first
    resp = client.chat(model="mock-triage-1",
                       messages=[{"role": "system", "content": SYSTEM_PROMPT},
                                 {"role": "user", "content": f"Ticket: {clean}\nCategory:"}])
    raw = resp["choices"][0]["message"]["content"]
    check = validate_category(raw)
    records.append({"ticket_id": row.ticket_id, "true": row.category,
                    "raw_reply": raw, "accepted": check["ok"], "predicted": check["category"]})

results = pd.DataFrame(records)
rejected = results[~results["accepted"]]
print(f"replies accepted by the validator : {results['accepted'].sum()} / {len(results)}")
print(f"replies rejected                  : {len(rejected)}\n")
rejected[["ticket_id", "raw_reply"]]

replies accepted by the validator : 37 / 40
replies rejected                  : 3



,ticket_id,raw_reply
6,TKT-1007,"Sorry, I am not sure about this one."
15,TKT-1016,"Sorry, I am not sure about this one."
39,TKT-1040,"Sorry, I am not sure about this one."


**Interpretation.** The validator caught the replies that were not categories at all and stopped them entering
the pipeline. Without it, the string `'Sorry, I am not sure about this one.'` would have been written into a
`category` column and quietly corrupted every downstream count.

A rejected reply is a **routing decision, not a crash**: send it to a human queue, or retry once with a
stricter prompt. What you must never do is coerce it into a category to keep the batch tidy.

In [9]:
ok = results[results["accepted"]]
accuracy = (ok["predicted"] == ok["true"]).mean()
print(f"accuracy on accepted replies : {accuracy:.3f}  (n = {len(ok)})")
print(f"coverage (share auto-handled): {len(ok) / len(results):.3f}")
print(f"routed to a human            : {len(rejected)}")

print("\nWhere the simulator disagreed with the true label:")
ok[ok["predicted"] != ok["true"]][["ticket_id", "true", "predicted"]].head(10)

accuracy on accepted replies : 0.865  (n = 37)
coverage (share auto-handled): 0.925
routed to a human            : 3

Where the simulator disagreed with the true label:


,ticket_id,true,predicted
3,TKT-1004,how_to,delivery
7,TKT-1008,how_to,billing
29,TKT-1030,how_to,delivery
31,TKT-1032,account,how_to
34,TKT-1035,billing,refund


**Interpretation.** Report **coverage and accuracy together**. A system that answers 90% of tickets at 85%
accuracy and escalates the rest is usually far more useful than one that answers everything at 78% — but you can
only make that trade-off visible if you measure both. Remember this is the keyword simulator, not a real LLM;
the *shape* of the evaluation is what transfers, not these particular scores.

---

## Part 5 — Terms of Service and ethics

Guardrails you can code are only half the obligation. These are the ones you have to *know*.

| Area | The working rule |
|---|---|
| **Personal data** | Send the minimum needed. Redact identifiers. Under the DPDP Act, purpose limitation and consent are legal duties, not preferences. |
| **Confidential data** | Client contracts, unreleased financials and source code do not go into a third-party API without written approval. |
| **Provider terms** | No reselling raw model output as your own service where prohibited; respect rate limits; no scraping the API. |
| **Training on your data** | Check whether your tier trains on submitted content. Enterprise tiers usually do not; free tiers often do. |
| **Attribution** | Disclose AI-generated content where a reader would reasonably expect a human wrote it. |
| **Accountability** | "The model decided" is not a defence. A named human owns every consequential automated decision. |

Three failures worth naming plainly, because all three are common:

1. **Pasting a customer database into a chat window to "clean it up."** That is a disclosure to a third party, and no consent covered it.
2. **Shipping model output as fact.** Session 29 showed hallucination is intrinsic. Ungrounded output needs verification before it reaches a customer.
3. **Automating a decision that affects a person's money, job or access, with no appeal route.** Whatever the accuracy, someone must be able to contest it and reach a human.

---

## Exercises

### Exercise 1 — Which tickets were rejected, and why?

List the ticket IDs the validator rejected, and explain in one line what they have in common.

<details><summary>Solution</summary>

```python
print(list(rejected["ticket_id"]))
for tid in rejected["ticket_id"]:
    print(tid, "|", tickets.loc[tickets["ticket_id"] == tid, "ticket_text"].iloc[0][:80])
```

**Three tickets: TKT-1007, TKT-1016 and TKT-1040.** None contains a keyword the simulator knows, so it fell
back to its "not sure" string. What they share is that all three are *warranty and product-behaviour questions*
phrased without any of the matcher's trigger words — the cases a keyword rule has least signal on. In production
these are exactly the ones worth routing to a human rather than guessing.

</details>

In [10]:
# TODO - Exercise 1


### Exercise 2 — Cost of the system prompt

The system prompt is re-sent on every request. What share of total input tokens is it, and what would you save
by halving it?

<details><summary>Solution</summary>

```python
system_total = count_tokens(SYSTEM_PROMPT) * len(tickets)
share = system_total / total_input
print(f"{system_total} of {total_input} input tokens = {share:.1%}")
print(f"halving it saves {system_total // 2} tokens per pass")
```

**2,960 of 4,307 input tokens — 68.7%** of everything sent. Halving the system prompt would save **1,480 tokens
per pass**. On a repetitive classification job the instructions, not the data, are the dominant cost.

</details>

In [11]:
# TODO - Exercise 2


### Exercise 3 — A stricter retry policy

`call_with_retry` retries `TimeoutError` and `ConnectionError`. A colleague proposes retrying *every* exception
so the batch never dies. Show why that is wrong by naming two errors it would retry pointlessly.

<details><summary>Solution</summary>

```python
# Retrying these is always wasted work - they are deterministic failures:
#   401 Unauthorized  -> the key is wrong; it will be wrong on every attempt
#   400 Bad Request   -> the request is malformed; identical request, identical rejection
# Retrying them burns time and quota, and hides the real fault from whoever has to fix it.
```

A blanket `except Exception: retry` turns a clear, immediate error into a slow, confusing one. Worse, on a
billing or content-policy rejection it can repeat a request that should never have been sent at all. Retry only
failures that are genuinely transient.

</details>

In [12]:
# TODO - Exercise 3


---

## Recap, mapped to the learning objectives

**LO 1 — API anatomy and key safety.** Every provider takes a model, a `messages` list of roles, decoding
parameters, and returns text plus a `usage` block. Keys load from the environment, are masked in logs, and fail
loudly rather than leaking.

**LO 2 — Usage patterns.** Retry only transient failures, back off exponentially, cap attempts and re-raise
honestly at the end. Estimate cost before running a batch — here the system prompt alone was **68.7%** of all
input tokens.

**LO 3 — Guardrails and ethics.** Redact on the way in (9 identifiers removed, with no loss of task signal) and
validate on the way out (3 non-category replies caught before they reached the pipeline, leaving 92.5% coverage
at 0.865 accuracy). Report coverage alongside accuracy, and remember that a named human owns every consequential
decision.

The sentence to carry into session 13.1: **an LLM call is only production-ready when you can say what it costs,
what it does when it fails, and what it refuses to send.**